# ModelFlow input widgets

`modelwidget_input` contains Jupyter-friendly input widgets for editing the data used by a ModelFlow model, running alternative scenarios, and inspecting the resulting solutions interactively.

The module is intended to provide a small **scenario laboratory** around a ModelFlow model. A user can change selected exogenous variables through sliders, radio buttons, check boxes, or spreadsheet-like controls instead of editing a DataFrame directly. The changed DataFrame can then be passed to the model, solved, stored as a scenario, and compared with other solutions.

The module has two main layers.

## 1. A widget framework

Each input widget follows a common interface. A widget knows how to:

- expose a displayable Jupyter widget through `datawidget`,
- apply the values currently selected by the user to a pandas DataFrame with `update_df(df, current_per)`,
- restore its initial state with `reset()`.

Widgets can either be simple **leaf widgets** or **container widgets**.

Leaf widgets directly change model variables. Container widgets combine several child widgets and forward updates and resets to them.

The common interface makes it possible to construct larger interfaces without the calling code needing to know the exact widget type.

## 2. Scenario execution and result inspection

The `updatewidget` class connects a widget tree to a ModelFlow model.

A typical workflow is:

1. Start from a baseline DataFrame.
2. Let the input widgets modify selected variables.
3. Run the ModelFlow model with the modified DataFrame.
4. Store the resulting solution as a named scenario.
5. Compare stored scenarios and selected variables interactively.

Stored model solutions can be inspected with `keep_plot_widget`, which uses the solutions normally kept in `mmodel.keep_solutions`.

## Widget definition format

Widgets are normally described by a small recursive Python structure:

```python
['widgettype', {'content': ...}]
```

The definition is converted into a live widget with:

```python
w = make_widget(widgetdef)
```

Complex interfaces can be built by nesting widget definitions inside container widgets.

For example:

```python
basedef = [
    'base',
    {
        'content': [
            slidedef,
            radiodef,
            checkdef,
        ]
    }
]

w = make_widget(basedef)
w
```

## Main widget types

### `slide`

Creates one or more sliders. A slider can change a variable by setting a value, adding a value, applying a percentage change, or using one of the related operators implemented by the widget.

Each slider has an operator in `'op'` (default `'='`) which decides how the slider value changes the variable in the selected periods:

| `op` | Effect | Periods |
|---|---|---|
| `'='` | set the variable to the value | all selected |
| `'=impulse'` | set the variable to the value | first selected |
| `'=start-'` | set the variable to the value | all periods *before* the first selected |
| `'+'` | add the value to the variable | all selected |
| `'+impulse'` | add the value to the variable | first selected |
| `'%'` | change the variable by value percent: × (1 + value/100) | all selected |
| `'%growth'` | let the variable grow value percent per period from the period before (needs `'start_value'` if there is no earlier period) | all selected |
| `'%of'` | set the variable to value percent of `'divisor'`: value/100 × divisor | all selected |
| `'+%of'` | add value percent of `'divisor'` to the variable | all selected |
| `'%of_impulse'` | as `'%of'` | first selected |
| `'+%of_impulse'` | as `'+%of'` | first selected |

The `%of` operators need `'divisor'`, the name of a variable, typically GDP, so the slider is in percent of GDP. `'var'` can hold several variables separated by spaces; the operator is applied to each of them.

This is useful for model shocks such as interest-rate changes, tax changes, productivity shocks, or changes in external assumptions.

### `sumslide`

Creates a group of sliders whose values are constrained to sum to a specified total.

This is useful for shares and allocations, for example portfolio shares, expenditure shares, or the maturity composition of debt issuance.

### `radio`

Lets the user select one alternative from a number of choices. The widget writes indicator variables, normally as zeros and ones, to the DataFrame.

### `check`

Provides on/off selections which can be mapped to model variables.

### `sheet`

Provides a spreadsheet-like editor based on `ipydatagrid`.

This can be useful when several values across variables or periods should be edited directly.

### `base`

A container which places several widgets together in a vertical layout.

### `tab`

A container which organizes several widget definitions in tabs or in an accordion.

## Data and period assumptions

The widgets are designed around the normal ModelFlow data structure:

- data are stored in a pandas DataFrame,
- periods are in the DataFrame index,
- model variables are DataFrame columns,
- `current_per` specifies the periods currently used for the experiment.

Most widgets therefore make changes using expressions equivalent to:

```python
df.loc[current_per, variable]
```

Some operators use the first selected period specially, for example when constructing an impulse.

## Using widgets with ModelFlow

A basic ModelFlow session normally begins by importing the main `model` class.

The widget definitions can then be constructed independently of the model. Once a model and its baseline data are available, `updatewidget` can connect the input interface to the model.

A minimal pattern is:

```python
w = make_widget(widgetdef)
ui = updatewidget(mmodel, w, varpat='*')
ui
```

The exact widget definitions will normally depend on the model: which variables should be changed, which values and ranges are sensible, and how the resulting scenarios should be presented.

The following cells first import the ModelFlow model class and the widget tools. Later cells can define a small example widget and connect it to a model.


In [10]:
import sys
if sys.platform == 'emscripten':  # only in the browser (JupyterLite): install ModelFlow, see mfsetup.py
    %run mfsetup.py
    await install_modelflow()

In [11]:
from modelclass import model

In [12]:
from modelwidget_input import make_widget, updatewidget

In [13]:
mpak,bline = model.modelload('data/pak.pcim',start=2023,end=2040,run=True,keep='Baseline')

Zipped file read:  data\pak.pcim


In [14]:
# Government spending variables to be held constant
GGexp='PAKGGREVOTHRCN PAKGGEXPCAPTCN PAKGGEXPGNFSCN PAKGGEXPTRNSCN PAKGGEXPOTHRCN PAKNEGDIFGOVXN PAKBXFSTREMTCD PAKBMFSTREMTCD'
mpak.basedf  = mpak.fix(bline,f'{GGexp}') # Freeze other spending levels

The folowing variables are fixed
PAKGGREVOTHRCN
PAKGGEXPCAPTCN
PAKGGEXPGNFSCN
PAKGGEXPTRNSCN
PAKGGEXPOTHRCN
PAKNEGDIFGOVXN
PAKBXFSTREMTCD
PAKBMFSTREMTCD


In [15]:
co2_common = {'value': 0.0, 'min': -50.0, 'max': 50.0, 'step': 1.0, 'op': '+'}
gov_common = {'value': 0.0, 'min': -2.0, 'max': 2.0, 'step': 0.1, 'op': '+%of', 'divisor': 'PAKNYGDPMKTPCN'}

co2_slidedef = ['slide', {
    'heading': 'CO2 tax rate changes ',
    'content': {
        'Coal': {'var': 'PAKGGREVCO2CER', **co2_common},
        'Gas':  {'var': 'PAKGGREVCO2GER', **co2_common},
        'Oil':  {'var': 'PAKGGREVCO2OER', **co2_common},
    }
}]

tfp_slidedef = ['slide', {
    'heading': 'Total factor productivity, % change ',
    'content': {
        'Total factor productivity': {'var': 'PAKNYGDPTFP', 'value': 0.0, 'min': -50.0, 'max': 50.0, 'step': 1.0, 'op': '%'},
    }
}]

gov_slidedef = ['slide', {
    'heading': 'Government expenditure, % of GDP ',
    'content': {
        'Transfers to households':              {'var': 'PAKGGEXPTRNSCN_X', **gov_common},
        'Gov. expenditure on investment goods': {'var': 'PAKGGEXPCAPTCN_X', **gov_common},
    }
}]

tabdef = ['tab', {
    'content': [
        ('CO2 tax',                    co2_slidedef),
        ('Total factor productivity',  tfp_slidedef),
        ('Government expenditure',     gov_slidedef),
    ]
}]

winput = make_widget(tabdef)


In [16]:
ui = updatewidget(mpak, winput, varpat='*')
ui